In [6]:
"""
Smart Factory — Industrial Analytics Dashboard  v5.0  (flat-schema edition)
Run:  python app.py  →  http://127.0.0.1:8053

pip install dash dash-bootstrap-components plotly pandas numpy scipy statsmodels
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════

CSV_PATH = r'Data\final_dataset.csv'

RISK_W    = {"vibration": 0.40, "pressure": 0.30, "volt": 0.30}
ANOMALY_Z = 3.0

C = {
    "bg":      "#F0F2F6",
    "card":    "#FFFFFF",
    "border":  "#E3EBF6",
    "primary": "#2C7BE5",
    "success": "#00C896",
    "warning": "#F6C343",
    "danger":  "#E63757",
    "purple":  "#6B5CE7",
    "teal":    "#00B8D9",
    "text":    "#12263F",
    "sub":     "#95AAC9",
}

RISK_COLORS = {"Low": C["success"], "Medium": C["warning"], "High": C["danger"]}
RISK_EMOJIS = {"Low": "🟢 Low", "Medium": "🟡 Medium", "High": "🔴 High"}

TELEM_LABELS = {
    "volt":      "Voltage (V)",
    "rotate":    "Rotation (rpm)",
    "pressure":  "Pressure (bar)",
    "vibration": "Vibration (mm/s)",
}

BASE_LAYOUT = dict(
    plot_bgcolor  = C["card"],
    paper_bgcolor = C["card"],
    font          = dict(family="Inter, Segoe UI, sans-serif", size=13, color=C["text"]),
    margin        = dict(l=16, r=16, t=50, b=16),
    hoverlabel    = dict(bgcolor=C["card"], font_size=13, bordercolor=C["border"]),
)

# ══════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════

def gl(**kw):
    return {**BASE_LAYOUT, **kw}

def norm01(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.0, index=s.index)

def safe_to_datetime(series):
    return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)

def unify_id(df, col="machineID"):
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    return df

def apply_filters(data, machine, model, start, end):
    mask = pd.Series(True, index=data.index)
    if machine != "ALL" and "machineID" in data.columns:
        mask &= data["machineID"] == str(machine)
    if model != "ALL" and "model" in data.columns:
        mask &= data["model"] == model
    if "datetime" in data.columns:
        if start:
            mask &= data["datetime"] >= pd.to_datetime(start)
        if end:
            mask &= data["datetime"] <= pd.to_datetime(end)
    return data.loc[mask]

def empty_fig(title="No Data Available"):
    return go.Figure().update_layout(**gl(title=title))

# ══════════════════════════════════════════════════════════════════════════
# DATA LOADING  ←  rewritten for flat schema (no prefixes)
# ══════════════════════════════════════════════════════════════════════════

print("Loading data ...")
_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"    Raw shape: {_raw.shape}")
print(f"    Columns  : {list(_raw.columns)}")

# ── Normalise column names (strip whitespace)
_raw.columns = _raw.columns.str.strip()

# ── datetime
if "datetime" in _raw.columns:
    _raw["datetime"] = safe_to_datetime(_raw["datetime"])
elif "last_maint_datetime" in _raw.columns:
    _raw["datetime"] = safe_to_datetime(_raw["last_maint_datetime"])
else:
    _raw["datetime"] = pd.NaT

# ── machineID  (try common variants)
for _mid_col in ["machineID", "machine_id", "MachineID", "machine"]:
    if _mid_col in _raw.columns:
        if _mid_col != "machineID":
            _raw.rename(columns={_mid_col: "machineID"}, inplace=True)
        break
else:
    _raw["machineID"] = "M1"          # fallback: treat whole dataset as one machine

unify_id(_raw)

# ── Core telemetry columns
TELEM_CORE = ["volt", "rotate", "pressure", "vibration"]
for c in TELEM_CORE:
    if c in _raw.columns:
        _raw[c] = pd.to_numeric(_raw[c], errors="coerce")

# ── model / age
if "model" not in _raw.columns and "model_encoded" in _raw.columns:
    _raw["model"] = "Model_" + _raw["model_encoded"].astype(str)
if "age" not in _raw.columns and "machine_age" in _raw.columns:
    _raw.rename(columns={"machine_age": "age"}, inplace=True)
if "age" in _raw.columns:
    _raw["age"] = pd.to_numeric(_raw["age"], errors="coerce")

# ── Build telem (= the main flat df)
telem_cols = ["machineID", "datetime"] + \
             [c for c in TELEM_CORE if c in _raw.columns] + \
             [c for c in ["model", "age"] if c in _raw.columns]
telem = _raw[telem_cols].copy().reset_index(drop=True)

# ── machines lookup
machines_cols = ["machineID"] + [c for c in ["model", "age"] if c in _raw.columns]
machines = (_raw[machines_cols]
            .drop_duplicates(subset=["machineID"])
            .reset_index(drop=True))

# ── errors synthetic: use error_count column if present
if "error_count" in _raw.columns:
    _err_rows = _raw[_raw["error_count"] > 0][["machineID", "datetime", "error_count"]].copy()
    # expand: one row per error event (capped at 5 to avoid explosion)
    _expanded = []
    for _, row in _err_rows.iterrows():
        n = min(int(row["error_count"]), 5)
        for i in range(1, n + 1):
            _expanded.append({"machineID": row["machineID"],
                               "datetime":  row["datetime"],
                               "errorID":   f"error{i}"})
    errors = pd.DataFrame(_expanded) if _expanded else pd.DataFrame(
        columns=["machineID", "datetime", "errorID"])
else:
    errors = pd.DataFrame(columns=["machineID", "datetime", "errorID"])

# ── failures synthetic: use failure_flag column if present
if "failure_flag" in _raw.columns:
    failures = (_raw[_raw["failure_flag"] == 1][["machineID", "datetime"]]
                .copy().dropna(subset=["datetime"]))
    # Try to get failure component label
    if "comp" in _raw.columns:
        failures["failure"] = _raw.loc[failures.index, "comp"].fillna("unknown").values
    else:
        failures["failure"] = "failure"
    failures = failures.reset_index(drop=True)
elif "target" in _raw.columns:
    failures = (_raw[_raw["target"] == 1][["machineID", "datetime"]]
                .copy().dropna(subset=["datetime"]))
    failures["failure"] = "failure"
    failures = failures.reset_index(drop=True)
else:
    failures = pd.DataFrame(columns=["machineID", "datetime", "failure"])

print(f"    Telem:{len(telem):,}  Machines:{len(machines)}  "
      f"Errors:{len(errors):,}  Failures:{len(failures):,}")

# ── Master df
df = telem.copy()

# ── numeric coerce
for c in ["volt", "rotate", "pressure", "vibration", "age"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df.dropna(subset=["machineID"], inplace=True)
unify_id(df)

# ── Risk score
for c in ["volt", "rotate", "pressure", "vibration"]:
    if c in df.columns:
        df[f"{c}_n"] = norm01(df[c])

df["risk_score"] = 0.0
active_w = {c: w for c, w in RISK_W.items() if f"{c}_n" in df.columns}
total_w  = sum(active_w.values()) or 1
for c, w in active_w.items():
    df["risk_score"] += df[f"{c}_n"].fillna(0) * (w / total_w)

df["risk_cat"] = pd.cut(
    df["risk_score"],
    bins=[-0.001, 0.333, 0.666, 1.001],
    labels=["Low", "Medium", "High"],
)
df["risk_label"] = df["risk_cat"].map(RISK_EMOJIS)
df["year_month"]  = df["datetime"].dt.to_period("M").astype(str)

# ── Anomaly flags
for c in ["volt", "rotate", "pressure", "vibration"]:
    if c in df.columns:
        z = (df[c] - df[c].mean()) / (df[c].std() + 1e-9)
        df[f"{c}_anomaly"] = z.abs() > ANOMALY_Z

anomaly_cols = [f"{c}_anomaly" for c in ["volt","rotate","pressure","vibration"]
                if f"{c}_anomaly" in df.columns]
df["any_anomaly"] = df[anomaly_cols].any(axis=1) if anomaly_cols else False

print(f"    Master df: {len(df):,} rows x {len(df.columns)} cols")

# ══════════════════════════════════════════════════════════════════════════
# KPIs
# ══════════════════════════════════════════════════════════════════════════

n_machines         = df["machineID"].nunique()
total_error_events = int(errors["errorID"].notna().sum()) if not errors.empty else 0
unique_error_types = int(errors["errorID"].nunique())     if not errors.empty else 0
total_failures     = len(failures)
avg_risk_pct       = round(float(df["risk_score"].mean()) * 100, 1)
asset_health       = round(100 - avg_risk_pct, 1)
avg_age            = round(float(df["age"].dropna().mean()), 1) if "age" in df.columns else "N/A"
n_anomalies        = int(df["any_anomaly"].sum())
n_high_risk        = int((df.groupby("machineID")["risk_score"].mean() > 0.666).sum())

DATE_MIN = df["datetime"].min() if df["datetime"].notna().any() else None
DATE_MAX = df["datetime"].max() if DATE_MIN is not None else None

# ══════════════════════════════════════════════════════════════════════════
# AGGREGATIONS
# ══════════════════════════════════════════════════════════════════════════

_agg = {c: "mean" for c in ["risk_score","volt","rotate","pressure","vibration"] if c in df.columns}
_agg["any_anomaly"] = "sum"
machine_stats = df.groupby("machineID", observed=True).agg(_agg).reset_index()
machine_stats.rename(columns={"any_anomaly": "anomaly_count"}, inplace=True)
unify_id(machine_stats)

if "model" in df.columns:
    machine_stats = machine_stats.merge(
        df[["machineID","model"]].drop_duplicates("machineID"), on="machineID", how="left")
if "age" in df.columns:
    machine_stats = machine_stats.merge(
        df[["machineID","age"]].drop_duplicates("machineID"), on="machineID", how="left")

if not errors.empty:
    err_per_m = (errors.groupby("machineID")
                 .agg(error_events=("errorID","count"), unique_errors=("errorID","nunique"))
                 .reset_index())
else:
    err_per_m = pd.DataFrame(columns=["machineID","error_events","unique_errors"])

if not failures.empty:
    fail_per_m = failures.groupby("machineID").size().reset_index(name="failure_count")
else:
    fail_per_m = pd.DataFrame(columns=["machineID","failure_count"])

err_fail_scatter = err_per_m.merge(fail_per_m, on="machineID", how="outer").fillna(0)
if "model" in machine_stats.columns:
    err_fail_scatter = err_fail_scatter.merge(
        machine_stats[["machineID","model"]], on="machineID", how="left")

TELEM_COLS = [c for c in ["volt","rotate","pressure","vibration"] if c in df.columns]
corr_df = (df[TELEM_COLS].dropna().corr()
           .rename(columns=TELEM_LABELS, index=TELEM_LABELS))

risk_dist = (df["risk_label"].value_counts()
             .reset_index().rename(columns={"risk_label":"Risk Level","count":"Count"}))

top10_risk = (machine_stats.nlargest(10, "risk_score")
              [["machineID","risk_score"] + (["model"] if "model" in machine_stats.columns else [])]
              .sort_values("risk_score"))

top10_anomaly = (machine_stats.nlargest(10, "anomaly_count")
                 [["machineID","anomaly_count"] + (["model"] if "model" in machine_stats.columns else [])]
                 .sort_values("anomaly_count"))

if not failures.empty and failures["datetime"].notna().any():
    cum_fail = (failures.dropna(subset=["datetime"])
                .sort_values("datetime")
                .assign(cum=lambda x: range(1, len(x)+1))
                [["datetime","cum"]])
else:
    cum_fail = pd.DataFrame()

err_type_dist = pd.DataFrame()
if not errors.empty:
    err_type_dist = (errors["errorID"].value_counts()
                     .head(5).reset_index()
                     .rename(columns={"errorID":"Error Type","count":"Events"}))

fail_type_dist = pd.DataFrame()
if not failures.empty and "failure" in failures.columns:
    fail_type_dist = (failures["failure"].value_counts()
                      .reset_index()
                      .rename(columns={"failure":"Failure Type","count":"Count"}))

avg_risk_model = pd.DataFrame()
if "model" in df.columns:
    avg_risk_model = (df.groupby("model", observed=True)["risk_score"]
                      .mean().reset_index()
                      .rename(columns={"risk_score":"avg_risk"})
                      .sort_values("avg_risk", ascending=False))

fail_trend = pd.DataFrame()
if not failures.empty and failures["datetime"].notna().any():
    fail_trend = (failures.dropna(subset=["datetime"])
                  .assign(month=lambda x: x["datetime"].dt.to_period("M").astype(str))
                  .groupby("month").size().reset_index(name="failures")
                  .sort_values("month"))

MODEL_OPTIONS   = [{"label":"All Models","value":"ALL"}] + \
                  ([{"label":m,"value":m} for m in sorted(df["model"].dropna().unique())]
                   if "model" in df.columns else [])
MACHINE_OPTIONS = [{"label":"All Machines","value":"ALL"}] + \
                  [{"label":str(m),"value":str(m)} for m in sorted(df["machineID"].dropna().unique())]
METRIC_OPTIONS  = [{"label":v,"value":k} for k,v in TELEM_LABELS.items() if k in df.columns]

print("Aggregations ready.\n")

# ══════════════════════════════════════════════════════════════════════════
# STATIC FIGURES
# ══════════════════════════════════════════════════════════════════════════

_risk_cmap = {v: RISK_COLORS[k] for k, v in RISK_EMOJIS.items()}
_GCF = {"displayModeBar": False}

# ── Top-10 risk ranking
fig_risk_rank = px.bar(
    top10_risk, x="risk_score", y="machineID", orientation="h",
    title="Top 10 High-Risk Machines",
    color="risk_score",
    color_continuous_scale=[[0,C["success"]],[0.5,C["warning"]],[1,C["danger"]]],
    text=top10_risk["risk_score"].round(3).astype(str),
)
fig_risk_rank.update_traces(textposition="outside")
fig_risk_rank.update_layout(**gl(coloraxis_showscale=False,
                                  xaxis_title="Avg Risk Score (0-1)", yaxis_title="Machine ID"))

# ── Correlation heatmap
_cz = corr_df.values.round(2)
fig_corr = go.Figure(go.Heatmap(
    z=_cz, x=list(corr_df.columns), y=list(corr_df.index),
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=_cz, texttemplate="%{text}", showscale=True,
    hovertemplate="%{y} x %{x}: %{z:.2f}<extra></extra>",
))
fig_corr.update_layout(**gl(title="Telemetry Correlation Matrix"))

# ── Cumulative failures
if not cum_fail.empty:
    fig_cumul = px.line(cum_fail, x="datetime", y="cum",
                        title="Cumulative Failures Over Time",
                        color_discrete_sequence=[C["primary"]])
    fig_cumul.update_traces(line_width=2.5,
                            hovertemplate="%{x|%Y-%m-%d}<br>Failures: %{y}<extra></extra>")
    fig_cumul.update_layout(**gl(xaxis_title="Date", yaxis_title="Cumulative Failures"))
else:
    fig_cumul = empty_fig("No Failure Datetime Data")

# ── Fleet Risk Distribution donut
fig_risk_dist = px.pie(
    risk_dist, names="Risk Level", values="Count",
    title="Fleet Risk Distribution",
    hole=0.44, color="Risk Level", color_discrete_map=_risk_cmap,
)
fig_risk_dist.update_traces(textposition="inside", textinfo="percent+label",
                             hovertemplate="%{label}<br>%{value:,} readings (%{percent})<extra></extra>")
fig_risk_dist.update_layout(**gl())

# ── Top-5 error types
if not err_type_dist.empty:
    fig_err_type = px.bar(
        err_type_dist.sort_values("Events"), x="Events", y="Error Type", orientation="h",
        title="Top 5 Error Types",
        color="Events",
        color_continuous_scale=[[0,C["sub"]],[1,C["warning"]]],
        text="Events",
    )
    fig_err_type.update_traces(textposition="outside")
    fig_err_type.update_layout(**gl(coloraxis_showscale=False, xaxis_title="Event Frequency"))
else:
    fig_err_type = empty_fig("No Error Data")

# ── Average Risk per Model
if not avg_risk_model.empty:
    fig_risk_model = px.bar(
        avg_risk_model, x="model", y="avg_risk",
        title="Average Risk Score per Model",
        color="avg_risk",
        color_continuous_scale=[[0,C["success"]],[0.5,C["warning"]],[1,C["danger"]]],
        text=avg_risk_model["avg_risk"].round(3).astype(str),
    )
    fig_risk_model.update_traces(textposition="outside")
    fig_risk_model.update_layout(**gl(coloraxis_showscale=False,
                                       xaxis_title="Model", yaxis_title="Avg Risk Score"))
else:
    fig_risk_model = empty_fig("No Model Data")

# ── Failure Type Treemap
if not fail_type_dist.empty:
    fig_fail_type = px.treemap(
        fail_type_dist,
        path=["Failure Type"],
        values="Count",
        title="Failure Type Breakdown",
        color="Count",
        color_continuous_scale=[[0, C["warning"]], [1, C["danger"]]],
    )
    fig_fail_type.update_traces(
        textinfo="label+value+percent root",
        hovertemplate="<b>%{label}</b><br>Count: %{value:,}<br>%{percentRoot:.1%} of total<extra></extra>",
    )
    fig_fail_type.update_layout(**gl(coloraxis_showscale=False))
else:
    fig_fail_type = empty_fig("No Failure Type Data")

# ── Anomaly Bubble Chart
_bubble_data = machine_stats.copy()
if "age" not in _bubble_data.columns:
    _bubble_data["age"] = 5

if not _bubble_data.empty and _bubble_data["anomaly_count"].sum() > 0:
    fig_anomaly = px.scatter(
        _bubble_data,
        x="anomaly_count",
        y="risk_score",
        size="age",
        size_max=45,
        color="model" if "model" in _bubble_data.columns else None,
        hover_name="machineID",
        hover_data={
            "anomaly_count": True,
            "risk_score": ":.3f",
            "age": True,
            "model": True if "model" in _bubble_data.columns else False,
        },
        title=f"Anomaly Count vs Risk Score  (bubble size = Machine Age)  |  z > {ANOMALY_Z}",
        opacity=0.75,
    )
    fig_anomaly.update_layout(
        **gl(xaxis_title="Anomaly Count", yaxis_title="Avg Risk Score (0-1)"),
        legend_title_text="Model",
    )
else:
    fig_anomaly = empty_fig("No Anomalies Detected")

# ── Errors vs Failures Scatter + Quadrant Zones
if not err_fail_scatter.empty and "failure_count" in err_fail_scatter.columns:
    _ef = err_fail_scatter.copy()
    _ef["failure_count"] = _ef["failure_count"].astype(float)
    _ef["error_events"]  = _ef["error_events"].astype(float)

    med_err  = _ef["error_events"].median()
    med_fail = _ef["failure_count"].median()
    max_err  = _ef["error_events"].max() or 1
    max_fail = _ef["failure_count"].max() or 1

    _ef_hover = {c: True for c in ["machineID"] + (["model"] if "model" in _ef.columns else [])}

    fig_err_fail = px.scatter(
        _ef, x="error_events", y="failure_count",
        title="Errors vs Failures per Machine  —  Quadrant Analysis",
        color="failure_count",
        color_continuous_scale=[[0, C["sub"]], [1, C["danger"]]],
        size="failure_count", size_max=30, opacity=0.75,
        hover_data=_ef_hover, trendline="ols",
    )
    fig_err_fail.add_vline(
        x=med_err, line_dash="dash", line_color=C["sub"], line_width=1.5,
        annotation_text=f"Median errors: {med_err:.0f}",
        annotation_position="top left",
        annotation_font_color=C["sub"],
    )
    fig_err_fail.add_hline(
        y=med_fail, line_dash="dash", line_color=C["sub"], line_width=1.5,
        annotation_text=f"Median failures: {med_fail:.0f}",
        annotation_position="bottom right",
        annotation_font_color=C["sub"],
    )
    fig_err_fail.add_annotation(x=max_err*0.85, y=max_fail*0.92, text="🔴 Critical",
        font=dict(color=C["danger"],  size=13, family="Inter, sans-serif"), showarrow=False)
    fig_err_fail.add_annotation(x=max_err*0.85, y=max_fail*0.08, text="🟡 Early Warning",
        font=dict(color=C["warning"], size=12, family="Inter, sans-serif"), showarrow=False)
    fig_err_fail.add_annotation(x=max_err*0.10, y=max_fail*0.92, text="🟠 Surprise Failures",
        font=dict(color="#FF8C00",    size=12, family="Inter, sans-serif"), showarrow=False)
    fig_err_fail.add_annotation(x=max_err*0.10, y=max_fail*0.08, text="🟢 Safe",
        font=dict(color=C["success"], size=12, family="Inter, sans-serif"), showarrow=False)
    fig_err_fail.update_layout(
        **gl(coloraxis_showscale=False,
             xaxis_title="Total Error Events",
             yaxis_title="Total Failures"),
    )
else:
    fig_err_fail = empty_fig("No Error/Failure Data")

# ── Risk Score Violin per Model (or histogram fallback)
if "model" in df.columns and df["model"].notna().any():
    fig_risk_hist = px.violin(
        df, x="model", y="risk_score", color="model",
        box=True, points="outliers",
        title="Risk Score Distribution per Model",
        color_discrete_sequence=px.colors.qualitative.Bold,
    )
    fig_risk_hist.update_traces(
        hovertemplate="<b>%{x}</b><br>Risk Score: %{y:.3f}<extra></extra>",
        meanline_visible=True,
    )
    fig_risk_hist.update_layout(
        **gl(xaxis_title="Model", yaxis_title="Risk Score (0-1)"),
        showlegend=False,
    )
else:
    fig_risk_hist = px.histogram(
        df, x="risk_score", nbins=40,
        title="Risk Score Distribution (All Readings)",
        color_discrete_sequence=[C["primary"]],
    )
    fig_risk_hist.update_layout(**gl(xaxis_title="Risk Score (0-1)", yaxis_title="Count"))

# ── Monthly Failure Trend + Peak Highlight
if not fail_trend.empty:
    peak_idx   = fail_trend["failures"].idxmax()
    peak_month = fail_trend.loc[peak_idx, "month"]
    peak_val   = int(fail_trend.loc[peak_idx, "failures"])

    fig_fail_trend = px.line(
        fail_trend, x="month", y="failures",
        title="Monthly Failure Trend  —  Peak Month Highlighted",
        markers=True, color_discrete_sequence=[C["danger"]],
    )
    fig_fail_trend.update_traces(
        line_width=2.5, marker_size=6,
        fill="tozeroy", fillcolor="rgba(230,55,87,0.08)",
        hovertemplate="%{x}<br>Failures: %{y}<extra></extra>",
    )
    fig_fail_trend.add_trace(go.Scatter(
        x=[peak_month], y=[peak_val],
        mode="markers+text",
        marker=dict(color=C["danger"], size=16, symbol="circle",
                    line=dict(color="#fff", width=2)),
        text=[f"⚠️ Peak: {peak_val}"],
        textposition="top center",
        textfont=dict(color=C["danger"], size=12, family="Inter, sans-serif"),
        showlegend=False,
        hovertemplate=f"<b>Peak Month</b><br>{peak_month}<br>Failures: {peak_val}<extra></extra>",
    ))
    fig_fail_trend.add_annotation(
        x=peak_month, y=peak_val,
        text=f"<b>⚠️ Worst Month<br>{peak_month}</b>",
        showarrow=True, arrowhead=2, arrowcolor=C["danger"],
        ax=40, ay=-45,
        bgcolor=C["danger"],
        font=dict(color="#fff", size=11, family="Inter, sans-serif"),
        bordercolor=C["danger"], borderpad=5,
    )
    fig_fail_trend.update_layout(
        **gl(xaxis_title="Month", yaxis_title="Failure Count"),
        xaxis=dict(tickangle=-35),
    )
else:
    fig_fail_trend = empty_fig("No Failure Trend Data")

# ══════════════════════════════════════════════════════════════════════════
# LAYOUT HELPERS
# ══════════════════════════════════════════════════════════════════════════

_CARD = {"backgroundColor":C["card"],"borderRadius":"14px",
         "boxShadow":"0 1px 14px rgba(18,38,63,.07)"}
_LBL  = {"color":C["sub"],"fontSize":"11px","fontWeight":"700",
          "textTransform":"uppercase","letterSpacing":"0.06em",
          "display":"block","marginBottom":"6px"}
_DD   = {"width":"195px","fontSize":"14px"}

def kpi_card(icon, label, value, sub_label, sub_value, color):
    return html.Div(
        style={**_CARD,"padding":"20px 22px","flex":"1",
               "minWidth":"160px","borderTop":f"4px solid {color}"},
        children=[
            html.P(f"{icon}  {label}",
                   style={"color":C["sub"],"fontSize":"11px","fontWeight":"700",
                           "textTransform":"uppercase","letterSpacing":"0.06em","margin":"0 0 6px 0"}),
            html.H2(value, style={"color":color,"fontSize":"30px","fontWeight":"700",
                                   "margin":"0 0 4px 0","lineHeight":"1"}),
            html.P(f"{sub_label}: {sub_value}",
                   style={"color":C["sub"],"fontSize":"12px","margin":"0"}),
        ]
    )

def card(child, **extra):
    return html.Div(style={**_CARD,"padding":"12px 14px",**extra}, children=[child])

def row(*children, gap="20px", mb="24px"):
    return html.Div(style={"display":"flex","gap":gap,"marginBottom":mb,"flexWrap":"wrap"},
                    children=list(children))

def half(c):
    return html.Div(style={"flex":"1","minWidth":"320px"}, children=[c])

def badge(text, color):
    return html.Span(text, style={"backgroundColor":color,"color":"#fff",
                                   "borderRadius":"6px","padding":"2px 10px",
                                   "fontSize":"11px","fontWeight":"700",
                                   "marginLeft":"8px","verticalAlign":"middle"})

def section_title(title):
    return html.Div(style={"marginBottom":"16px","marginTop":"8px"}, children=[
        html.H3(title, style={"color":C["text"],"fontSize":"15px","fontWeight":"700",
                               "margin":"0","borderLeft":f"4px solid {C['primary']}",
                               "paddingLeft":"10px"}),
    ])

# ══════════════════════════════════════════════════════════════════════════
# APP LAYOUT
# ══════════════════════════════════════════════════════════════════════════

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP],
           title="Smart Factory Dashboard v5.0", suppress_callback_exceptions=True)
server = app.server

app.layout = html.Div(
    style={"backgroundColor":C["bg"],"minHeight":"100vh",
           "fontFamily":"Inter, Segoe UI, sans-serif","padding":"30px 36px"},
    children=[

        # Header
        html.Div(style={"marginBottom":"26px"}, children=[
            html.H1(
                ["Smart Factory — Industrial Analytics",
                 badge("v5.0",      C["primary"]),
                 badge("ML-Ready",  C["purple"]),
                 badge("Anomaly Detection", C["teal"])],
                style={"color":C["text"],"fontSize":"24px","fontWeight":"700","margin":"0 0 5px 0"},
            ),
            html.P(
                f"Dataset: {len(df):,} telemetry records  |  {n_machines} machines  |  "
                f"{unique_error_types} error types  |  {total_failures:,} failure events",
                style={"color":C["sub"],"fontSize":"13px","margin":"0"},
            ),
        ]),

        # KPI Strip
        row(
            kpi_card("💥","Failure Count", f"{total_failures:,}",
                     "Affected machines",
                     fail_per_m["machineID"].nunique() if not fail_per_m.empty else 0,
                     C["danger"]),
            kpi_card("⚠️","Error Events", f"{total_error_events:,}",
                     "Unique error types", unique_error_types, C["warning"]),
            kpi_card("🏭","Machines", f"{n_machines}",
                     "Avg age", f"{avg_age} yrs", C["primary"]),
            kpi_card("💚","Asset Health Score", f"{asset_health}%",
                     "High-risk machines", n_high_risk, C["success"]),
            kpi_card("🔬","Anomalies", f"{n_anomalies:,}",
                     "z threshold", ANOMALY_Z, C["teal"]),
        ),

        # Filter bar
        html.Div(
            style={**_CARD,"padding":"18px 24px","marginBottom":"24px",
                   "display":"flex","gap":"24px","alignItems":"flex-end","flexWrap":"wrap"},
            children=[
                html.Div([html.Label("Model",      style=_LBL),
                          dcc.Dropdown(id="dd-model", options=MODEL_OPTIONS,
                                       value="ALL", clearable=False, style=_DD)]),
                html.Div([html.Label("Machine ID", style=_LBL),
                          dcc.Dropdown(id="dd-machine", options=MACHINE_OPTIONS,
                                       value="ALL", clearable=False, style=_DD)]),
                html.Div([html.Label("Metric",     style=_LBL),
                          dcc.Dropdown(id="dd-metric", options=METRIC_OPTIONS,
                                       value=METRIC_OPTIONS[0]["value"] if METRIC_OPTIONS else None,
                                       clearable=False, style=_DD)]),
                html.Div([
                    html.Label("Date Range", style=_LBL),
                    dcc.DatePickerRange(id="dp-range", start_date=DATE_MIN, end_date=DATE_MAX,
                                       display_format="YYYY-MM-DD"),
                ]) if DATE_MIN else html.Div(),
            ],
        ),

        # ── Section 1: Telemetry
        section_title("Telemetry Monitor"),
        html.Div(style={"marginBottom":"24px"}, children=[card(dcc.Graph(id="telem-line", config=_GCF))]),

        # ── Section 2: Failure Analysis
        section_title("Failure Analysis"),
        row(half(card(dcc.Graph(figure=fig_cumul,      config=_GCF))),
            half(card(dcc.Graph(figure=fig_fail_trend, config=_GCF)))),
        row(half(card(dcc.Graph(figure=fig_err_type,   config=_GCF))),
            half(card(dcc.Graph(figure=fig_fail_type,  config=_GCF)))),

        # ── Section 3: Risk Intelligence
        section_title("Risk Intelligence"),
        row(half(card(dcc.Graph(figure=fig_risk_rank,  config=_GCF))),
            half(card(dcc.Graph(figure=fig_risk_dist,  config=_GCF)))),
        row(half(card(dcc.Graph(figure=fig_risk_model, config=_GCF))),
            half(card(dcc.Graph(figure=fig_risk_hist,  config=_GCF)))),
        html.Div(style={"marginBottom":"24px"}, children=[card(dcc.Graph(id="risk-scatter", config=_GCF))]),

        # ── Section 4: Anomaly & Correlation
        section_title("Anomaly Detection & Sensor Correlation"),
        row(half(card(dcc.Graph(figure=fig_anomaly,  config=_GCF))),
            half(card(dcc.Graph(figure=fig_corr,     config=_GCF)))),
        row(half(card(dcc.Graph(figure=fig_err_fail, config=_GCF)))),

        # Footer
        html.P(
            "Smart Factory Monitoring Platform v5.0  |  Risk-scored  |  Anomaly-detected  |  ML-Ready",
            style={"color":C["sub"],"fontSize":"12px","textAlign":"center","marginTop":"8px"},
        ),
    ],
)

# ══════════════════════════════════════════════════════════════════════════
# CALLBACKS
# ══════════════════════════════════════════════════════════════════════════

_BASE_IN = [Input("dd-model","value"), Input("dd-machine","value"), Input("dd-metric","value")]
if DATE_MIN:
    _BASE_IN += [Input("dp-range","start_date"), Input("dp-range","end_date")]

def unpack(args):
    if DATE_MIN:
        model, machine, metric, start, end = args
    else:
        model, machine, metric = args
        start = end = None
    return model, machine, metric, start, end


@app.callback(Output("telem-line","figure"), _BASE_IN)
def cb_telem(*args):
    model, machine, metric, start, end = unpack(args)
    if not metric or metric not in df.columns:
        return empty_fig("Select a metric")
    filtered = apply_filters(df, machine, model, start, end)
    if filtered.empty:
        return empty_fig("No data for current filters")
    monthly = (filtered.groupby("year_month", observed=True)[metric]
               .mean().reset_index().rename(columns={metric:"value"})
               .sort_values("year_month"))
    fleet_avg = float(df[metric].mean())
    label     = TELEM_LABELS.get(metric, metric)
    subtitle  = ("All Models" if model=="ALL" else model) + \
                ("" if machine=="ALL" else f"  |  Machine {machine}")
    fig = px.line(monthly, x="year_month", y="value",
                  title=f"Monthly Avg {label}  —  {subtitle}",
                  markers=True, color_discrete_sequence=[C["primary"]])
    fig.update_traces(line_width=2.5, marker_size=7,
                      hovertemplate="%{x}<br>Avg: %{y:.3f}<extra></extra>")
    fig.add_hline(y=fleet_avg, line_dash="dot", line_color=C["sub"],
                  annotation_text=f"Fleet avg: {fleet_avg:.2f}",
                  annotation_position="top right",
                  annotation_font_color=C["sub"])
    fig.update_layout(**gl(xaxis_title="Month", yaxis_title=label),
                      xaxis=dict(tickangle=-35))
    return fig


_SCATTER_IN = [Input("dd-model","value"), Input("dd-machine","value")]
if DATE_MIN:
    _SCATTER_IN += [Input("dp-range","start_date"), Input("dp-range","end_date")]

@app.callback(Output("risk-scatter","figure"), _SCATTER_IN)
def cb_scatter(*args):
    if DATE_MIN:
        model, machine, start, end = args
    else:
        model, machine = args
        start = end = None
    if "vibration" not in df.columns:
        return empty_fig("No vibration data")
    filtered = apply_filters(df, machine, model, start, end)
    if filtered.empty:
        return empty_fig("No data for selection")
    sample = filtered.sample(min(5_000, len(filtered)), random_state=42)
    hover  = {c:True for c in ["machineID"]
              + (["model"] if "model" in sample.columns else [])
              + (["age"]   if "age"   in sample.columns else [])}
    fig = px.scatter(
        sample, x="vibration", y="risk_score",
        color="risk_label", color_discrete_map=_risk_cmap,
        title=f"Vibration vs Risk Score  —  {'All Models' if model=='ALL' else model}",
        opacity=0.55, hover_data=hover,
        category_orders={"risk_label": list(RISK_EMOJIS.values())},
    )
    fig.update_layout(**gl(xaxis_title="Vibration (mm/s)", yaxis_title="Risk Score (0-1)"),
                      legend_title_text="Risk Level")
    return fig

# ══════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    app.run(debug=False, port=8053)

Loading data ...
    Raw shape: (875742, 57)
    Columns  : ['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration', 'error_count', 'failure_flag', 'target', 'last_maint_datetime', 'comp', 'maint_flag', 'days_since_maint', 'model', 'age', 'hour', 'dayofweek', 'month', 'is_weekend', 'volt_lag1', 'volt_lag3', 'rotate_lag1', 'rotate_lag3', 'pressure_lag1', 'pressure_lag3', 'vibration_lag1', 'vibration_lag3', 'error_count_lag1', 'error_count_lag3', 'volt_mean_3', 'volt_std_3', 'volt_max_3', 'rotate_mean_3', 'rotate_std_3', 'rotate_max_3', 'pressure_mean_3', 'pressure_std_3', 'pressure_max_3', 'vibration_mean_3', 'vibration_std_3', 'vibration_max_3', 'error_count_mean_3', 'error_count_std_3', 'error_count_max_3', 'volt_diff', 'rotate_diff', 'pressure_diff', 'vibration_diff', 'error_count_diff', 'recent_maint', 'log_days_since_maint', 'error_rate', 'error_trend', 'stress_index', 'power_stress', 'machine_age', 'model_encoded']
    Telem:875,742  Machines:100  Errors:3,914  Failure